# Hiperparámetros modelo Logistic regresion

![Duoc UC](https://upload.wikimedia.org/wikipedia/commons/thumb/a/aa/Logo_DuocUC.svg/960px-Logo_DuocUC.svg.png "Duoc UC")

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import randint
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
import json

## Predicción de Depresión en Estudiantes

Este cuaderno demuestra un flujo de trabajo de aprendizaje automático para predecir la depresión en estudiantes basándose en varias características. El proceso incluye la carga de datos, exploración inicial, división de datos, entrenamiento de un modelo base, ajuste de hiperparámetros utilizando `GridSearchCV` y `RandomizedSearchCV`, y la evaluación del modelo final.

### 1. Carga de Datos y Exploración Inicial

Primero, cargamos el conjunto de datos preprocesado de depresión estudiantil y realizamos una inspección inicial de su estructura y contenido.

In [ ]:
df = pd.read_csv('/content/Student_Depression_Dataset_codificado.csv')
df.head()

,cat__Gender_Male,cat__Sleep Duration_7-8 hours,cat__Sleep Duration_Less than 5 hours,cat__Sleep Duration_More than 8 hours,cat__Dietary Habits_Moderate,cat__Dietary Habits_Unhealthy,cat__Degree_B.Com,cat__Degree_B.Ed,cat__Degree_B.Tech,cat__Degree_BCA,...,cat__Degree_Other,bin__Have you ever had suicidal thoughts ?,bin__Family History of Mental Illness,num__Age,num__Academic Pressure,num__CGPA,num__Study Satisfaction,num__Work/Study Hours,num__Financial Stress,remainder__Depression
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.476593,1.345273,0.895400,-0.694638,-1.122130,-1.489063,1.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,-0.370683,-0.827109,-1.200717,1.510988,-1.122130,-0.793223,0.0
2,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.066087,-0.102982,-0.429182,1.510988,0.496561,-1.489063,0.0
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.450328,-0.102982,-1.412377,-0.694638,-0.852348,1.294297,1.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-0.165430,0.621145,0.321870,0.040570,-1.661693,-1.489063,0.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27817 entries, 0 to 27816
Data columns (total 25 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   cat__Gender_Male                            27817 non-null  float64
 1   cat__Sleep Duration_7-8 hours               27817 non-null  float64
 2   cat__Sleep Duration_Less than 5 hours       27817 non-null  float64
 3   cat__Sleep Duration_More than 8 hours       27817 non-null  float64
 4   cat__Dietary Habits_Moderate                27817 non-null  float64
 5   cat__Dietary Habits_Unhealthy               27817 non-null  float64
 6   cat__Degree_B.Com                           27817 non-null  float64
 7   cat__Degree_B.Ed                            27817 non-null  float64
 8   cat__Degree_B.Tech                          27817 non-null  float64
 9   cat__Degree_BCA                             27817 non-null  float64
 10  cat__Degre

La salida de `df.head()` muestra las primeras 5 filas del conjunto de datos, que contiene varias características categóricas (codificadas one-hot), binarias y numéricas. La variable objetivo, 'remainder__Depression', indica la presencia (1.0) o ausencia (0.0) de depresión.

`df.info()` confirma que hay 27,817 entradas y 25 columnas. Todas las columnas son de tipo `float64` y no tienen valores faltantes, lo que indica que los datos ya han sido preprocesados y limpiados. Esto es crucial para proceder con el entrenamiento del modelo sin pasos adicionales de imputación.

In [ ]:
# Definir las características (X) y la variable objetivo (y)
y = df.iloc[:, -1]   # La última columna como objetivo
X = df.drop(df.columns[-1], axis=1) # Todas las columnas excepto la última como características

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de y_train:", y_train.shape)
print("Dimensiones de X_test:", X_test.shape)
print("Dimensiones de y_test:", y_test.shape)

Dimensiones de X_train: (22253, 24)
Dimensiones de y_train: (22253,)
Dimensiones de X_test: (5564, 24)
Dimensiones de y_test: (5564,)


### 2. Preparación de Datos

Antes de entrenar cualquier modelo, el conjunto de datos se divide en características (X) y la variable objetivo (y). Posteriormente, estos se dividen en conjuntos de entrenamiento y prueba para evaluar el rendimiento del modelo con datos no vistos. `test_size=0.2` indica que el 20% de los datos se utilizará para la prueba, y `random_state=42` asegura la reproducibilidad de la división.

In [ ]:
print("X_train head:")
display(X_train.head())

X_train head:


,cat__Gender_Male,cat__Sleep Duration_7-8 hours,cat__Sleep Duration_Less than 5 hours,cat__Sleep Duration_More than 8 hours,cat__Dietary Habits_Moderate,cat__Dietary Habits_Unhealthy,cat__Degree_B.Com,cat__Degree_B.Ed,cat__Degree_B.Tech,cat__Degree_BCA,...,cat__Degree_MSc,cat__Degree_Other,bin__Have you ever had suicidal thoughts ?,bin__Family History of Mental Illness,num__Age,num__Academic Pressure,num__CGPA,num__Study Satisfaction,num__Work/Study Hours,num__Financial Stress
20754,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,-1.191694,-1.551237,0.813467,-1.429847,1.036124,-0.097383
8341,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.860834,-0.827109,-1.132440,1.510988,0.766342,1.294297
7582,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.245076,-1.551237,1.298237,0.775779,0.766342,-1.489063
812,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,-1.396947,-0.827109,0.588152,1.510988,0.226779,0.598457
13911,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,-0.370683,-0.827109,-0.811536,1.510988,-0.043003,-0.097383


In [ ]:
print("y_train head:")
display(y_train.head())

y_train head:


,remainder__Depression
20754,0.0
8341,1.0
7582,0.0
812,1.0
13911,0.0


Las salidas de `X_train.head()` y `y_train.head()` ofrecen un vistazo a los datos de entrenamiento después de la división. `X_train` contiene el conjunto de características, mientras que `y_train` contiene las etiquetas objetivo correspondientes. Esta verificación asegura que los datos han sido preparados correctamente para el entrenamiento del modelo.

In [ ]:
# Entrenar un modelo de Regresión Logística con parámetros por defecto como línea base
lr_default = LogisticRegression(random_state=42, solver='liblinear')
lr_default.fit(X_train, y_train)
y_pred_default = lr_default.predict(X_test)
accuracy_default = accuracy_score(y_test, y_pred_default)

### Búsqueda de hiperparámetros con GridSearchCV para Regresión Logística

Ahora, utilizaremos `GridSearchCV` para una búsqueda exhaustiva de hiperparámetros en el modelo de Regresión Logística. Esto nos permitirá encontrar la combinación óptima dentro de un rango más refinado, basándonos posiblemente en los resultados de `RandomizedSearchCV`.

In [ ]:
# Definir el modelo de Regresión Logística
lr_gs = LogisticRegression(random_state=42, solver='liblinear') # Usamos el mismo solver 'liblinear'

# Definir la cuadrícula de parámetros para GridSearchCV (ajustada para Regresión Logística)
# Nos enfocaremos en los valores de 'C' alrededor del mejor encontrado por RandomizedSearchCV
param_grid_lr_gs = {
    'C': np.logspace(-2, 2, 10), # Un rango más fino alrededor de los valores óptimos, 10 valores
    'penalty': ['l1', 'l2'] # L1 o L2 penalización
}

# Inicializar GridSearchCV para Regresión Logística
grid_search_lr = GridSearchCV(
    estimator=lr_gs,
    param_grid=param_grid_lr_gs,
    cv=5,       # Validación cruzada con 5 folds
    n_jobs=-1,  # Usar todos los núcleos disponibles
    verbose=2   # Mostrar detalles del progreso
)

# Ajustar GridSearchCV a los datos de entrenamiento
grid_search_lr.fit(X_train, y_train)

# Mostrar los mejores parámetros y la mejor puntuación
print("\nMejores parámetros para Logistic Regression (GridSearchCV):", grid_search_lr.best_params_)
print("Mejor puntuación (accuracy) para Logistic Regression (GridSearchCV):", grid_search_lr.best_score_)

# Evaluar el modelo con los mejores hiperparámetros en el conjunto de prueba
best_lr_model_gs = grid_search_lr.best_estimator_
y_pred_lr_gs = best_lr_model_gs.predict(X_test)
accuracy_lr_gs = accuracy_score(y_test, y_pred_lr_gs)
print("Precisión de Logistic Regression (GridSearchCV) en el conjunto de prueba:", accuracy_lr_gs)

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Mejores parámetros para Logistic Regression (GridSearchCV): {'C': np.float64(0.5994842503189409), 'penalty': 'l2'}
Mejor puntuación (accuracy) para Logistic Regression (GridSearchCV): 0.847705881016512
Precisión de Logistic Regression (GridSearchCV) en el conjunto de prueba: 0.8470524802300503


### Búsqueda de hiperparámetros con RandomSearchCV para Regresión Logística

Ahora, utilizaremos `RandomSearchCV` para una búsqueda aleatoria de hiperparámetros en el modelo de Regresión Logística

In [ ]:
# Definir el modelo de Regresión Logística
lr = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' es bueno para datasets pequeños y L1/L2 penalización

# Definir el espacio de búsqueda de hiperparámetros para Regresión Logística
param_dist_lr = {
    'C': np.logspace(-4, 4, 20), # Regularization strength
    'penalty': ['l1', 'l2'] # L1 or L2 regularization
}

# Configurar RandomizedSearchCV para Regresión Logística
random_search_lr = RandomizedSearchCV(
    estimator=lr,
    param_distributions=param_dist_lr,
    n_iter=20,  # Número de combinaciones de hiperparámetros a probar (menos que RF)
    cv=5,       # Validación cruzada con 5 folds
    verbose=2,  # Mostrar detalles del progreso
    random_state=42,
    n_jobs=-1   # Usar todos los núcleos disponibles
)

# Ejecutar la búsqueda de hiperparámetros
random_search_lr.fit(X_train, y_train)

# Mostrar los mejores hiperparámetros y el mejor score
print("\nMejores hiperparámetros para Regresión Logística:", random_search_lr.best_params_)
print("Mejor score de validación cruzada para Regresión Logística:", random_search_lr.best_score_)

# Evaluar el modelo con los mejores hiperparámetros en el conjunto de prueba
best_lr_model = random_search_lr.best_estimator_
y_pred_lr = best_lr_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Precisión de la Regresión Logística en el conjunto de prueba:", accuracy_lr)

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Mejores hiperparámetros para Regresión Logística: {'penalty': 'l2', 'C': np.float64(0.615848211066026)}
Mejor score de validación cruzada para Regresión Logística: 0.847705881016512
Precisión de la Regresión Logística en el conjunto de prueba: 0.8470524802300503


### Recuperación de los Mejores Parámetros y Comparación de Modelos

In [ ]:
# Mejores parámetros de RandomizedSearchCV
print("Mejores parámetros de Regresión Logística (RandomizedSearchCV):", random_search_lr.best_params_)

# Mejores parámetros de GridSearchCV
print("Mejores parámetros de Regresión Logística (GridSearchCV):", grid_search_lr.best_params_)

# Entrenar un modelo final con los mejores parámetros de RandomizedSearchCV
best_lr_random_model = LogisticRegression(**random_search_lr.best_params_, random_state=42, solver='liblinear')
best_lr_random_model.fit(X_train, y_train)
y_pred_random = best_lr_random_model.predict(X_test)
accuracy_random = accuracy_score(y_test, y_pred_random)
print(f"\nPrecisión del modelo final (RandomizedSearchCV) en el conjunto de prueba: {accuracy_random:.4f}")

# Entrenar un modelo final con los mejores parámetros de GridSearchCV
best_lr_grid_model = LogisticRegression(**grid_search_lr.best_params_, random_state=42, solver='liblinear')
best_lr_grid_model.fit(X_train, y_train)
y_pred_grid = best_lr_grid_model.predict(X_test)
accuracy_grid = accuracy_score(y_test, y_pred_grid)
print(f"Precisión del modelo final (GridSearchCV) en el conjunto de prueba: {accuracy_grid:.4f}")

# Comparación de rendimientos
print("\n--- Comparación de Rendimiento ---")
print(f"Accuracy por defecto: {accuracy_default:.4f}")
print(f"Accuracy RandomizedSearchCV: {accuracy_random:.4f}")
print(f"Accuracy GridSearchCV:     {accuracy_grid:.4f}")

if accuracy_grid > accuracy_random:
    print("GridSearchCV obtuvo una mejor precisión.")
elif accuracy_random > accuracy_grid:
    print("RandomizedSearchCV obtuvo una mejor precisión.")
else:
    print("Ambos métodos obtuvieron la misma precisión.")

Mejores parámetros de Regresión Logística (RandomizedSearchCV): {'penalty': 'l2', 'C': np.float64(0.615848211066026)}
Mejores parámetros de Regresión Logística (GridSearchCV): {'C': np.float64(0.5994842503189409), 'penalty': 'l2'}

Precisión del modelo final (RandomizedSearchCV) en el conjunto de prueba: 0.8471
Precisión del modelo final (GridSearchCV) en el conjunto de prueba: 0.8471

--- Comparación de Rendimiento ---
Accuracy por defecto: 0.8472
Accuracy RandomizedSearchCV: 0.8471
Accuracy GridSearchCV:     0.8471
Ambos métodos obtuvieron la misma precisión.


### Cómo usar el hiperparámetro `best_lr_random_model`

Una vez que hemos entrenado `best_lr_random_model` con los mejores hiperparámetros encontrados por `RandomizedSearchCV` y `GridSearchCV`, este modelo está listo para realizar predicciones sobre nuevos datos.

**Recuerda:** Los nuevos datos sobre los que quieras predecir deben tener el mismo formato (mismas columnas y preprocesamiento, por ejemplo, escalado si se aplicó) que los datos de entrenamiento (`X_train`).

Aquí te mostramos cómo puedes usarlo para hacer predicciones:

In [ ]:
# Acceder a los mejores parámetros de RandomizedSearchCV
best_hyperparameters_random = random_search_lr.best_params_
print("\nLos mejores hiperparámetros encontrados por RandomizedSearchCV son:")
for param, value in best_hyperparameters_random.items():
    print(f"  {param}: {value}")

# Acceder a los mejores parámetros de GridSearchCV
best_hyperparameters_grid = grid_search_lr.best_params_
print("\nLos mejores hiperparámetros encontrados por GridSearchCV son:")
for param, value in best_hyperparameters_grid.items():
    print(f"  {param}: {value}")

# --- Exportar a archivos JSON ---
output_filename_random = 'best_logistic_regression_hyperparameters_random.json'
with open(output_filename_random, 'w') as f:
    # Convertir los valores de numpy a tipos nativos de Python para la serialización JSON
    serializable_params_random = {k: v.item() if isinstance(v, np.generic) else v for k, v in best_hyperparameters_random.items()}
    json.dump(serializable_params_random, f, indent=4)
print(f"\nLos mejores hiperparámetros de RandomizedSearchCV también se han guardado en '{output_filename_random}'")

output_filename_grid = 'best_logistic_regression_hyperparameters_grid.json'
with open(output_filename_grid, 'w') as f:
    # Convertir los valores de numpy a tipos nativos de Python para la serialización JSON
    serializable_params_grid = {k: v.item() if isinstance(v, np.generic) else v for k, v in best_hyperparameters_grid.items()}
    json.dump(serializable_params_grid, f, indent=4)
print(f"Los mejores hiperparámetros de GridSearchCV también se han guardado en '{output_filename_grid}'")


Los mejores hiperparámetros encontrados por RandomizedSearchCV son:
  penalty: l2
  C: 0.615848211066026

Los mejores hiperparámetros encontrados por GridSearchCV son:
  C: 0.5994842503189409
  penalty: l2

Los mejores hiperparámetros de RandomizedSearchCV también se han guardado en 'best_logistic_regression_hyperparameters_random.json'
Los mejores hiperparámetros de GridSearchCV también se han guardado en 'best_logistic_regression_hyperparameters_grid.json'
